If no result is found, we can potentially automate the process.

Our LLM classifies `kein ergebnis` cases as `drittauskunft`, so the positive prediction set should contain them. The goal here is to apply simple regexes for `kein ergebnis` and check how many `dritt=True` cases match that pattern.

Example samples from Notion:

- `die Anfrage lieferte kein verwertbares Ergebnis Das BZST teilt mit, dass ... zu keinem Ergebnis geführt hat`
- `Der Abruf hat zu keinem Ergebnis geführt`

In [3]:
# 1) Get data from egvp intents where we sent intent as drittauskunft

In [4]:
import os
import json
from ast import literal_eval
from datetime import datetime, timedelta

import pandas as pd

import numpy as np
from IPython.display import clear_output

from python_utilities.db_connection import DbConnection

analytics_db = DbConnection('ANALYTICS', 'PROD_RDS')


INFO [2026-06-23 16:59:11] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file


In [5]:
start_date = "2026-05-01"
end_date = "2026-07-01"

In [7]:
egvp_intents = analytics_db.sql_to_df(f"""
    SELECT *
    FROM egvp_intents
    WHERE created_at >= '{start_date}'
    AND created_at <= '{end_date}'
""")


In [8]:
egvp_intents

,id,ticket_uuid,attachment_id,egvp_id,created_at,intents,ready_for_automation,status,retry,target_type,updated_at
0,8700,d8cf9871-8e85-5634-84db-4ddb0c2cf93a,61606726,NRW_B217776028864429abc58ae-eadc-4b7e-955d-f3e...,2026-05-01 05:58:20,"[{""params"": {""slug"": ""196961520455"", ""debtor_n...",1,sent,0,egvp,2026-05-01 05:58:20
1,8701,87677926-c87b-5e70-a601-719100a646c2,61607185,NRW_B217776071743390222069b-13d9-4c6d-9833-989...,2026-05-01 05:58:21,"[{""params"": {}, ""channel"": ""egvp"", ""attachment...",0,sent,0,egvp,2026-05-01 05:58:21
2,8702,87677926-c87b-5e70-a601-719100a646c2,61607184,NRW_B217776071743390222069b-13d9-4c6d-9833-989...,2026-05-01 05:58:21,"[{""params"": {}, ""channel"": ""egvp"", ""attachment...",0,sent,0,egvp,2026-05-01 05:58:21
3,8703,02fc93f1-84eb-5518-a18a-53176d204f2b,61608024,NRW_B21777615111200bfe48a29-aaa4-4a06-8072-521...,2026-05-01 07:29:28,"[{""params"": {}, ""channel"": ""egvp"", ""attachment...",0,sent,0,egvp,2026-05-01 07:29:28
4,8704,4a465ef7-deca-5738-86b5-d9352074a439,61608021,NRW_B2177761258996528056741-734e-42fe-bf1d-590...,2026-05-01 07:29:30,"[{""params"": {}, ""channel"": ""egvp"", ""attachment...",0,sent,0,egvp,2026-05-01 07:29:30
...,...,...,...,...,...,...,...,...,...,...,...
22997,43432,86012867-dd35-5091-8c7a-ac6eb0889906,69426307,NRW_B217822157142745968982a-4e1e-4666-8904-f7a...,2026-06-23 13:24:39,"[{""params"": {}, ""channel"": ""egvp"", ""attachment...",0,sent,0,egvp,2026-06-23 13:24:39
22998,43433,902b480b-e05b-5de3-8abc-92c93c8562e5,69426299,NRW_B2178221256977330a47d9c-df1b-40be-bb6b-cee...,2026-06-23 13:24:40,"[{""params"": {}, ""channel"": ""egvp"", ""attachment...",0,sent,0,egvp,2026-06-23 13:24:40
22999,43434,17c2b57c-d3e0-5341-bbad-af6e74ecf060,69427232,NRW_B21782218042522b1926ee5-7aa7-4888-881c-a4c...,2026-06-23 14:24:46,"[{""params"": {}, ""channel"": ""egvp"", ""attachment...",0,sent,0,egvp,2026-06-23 14:24:46
23000,43435,a4359187-2ba2-540f-8179-0ff82e1adab3,69427175,NRW_B21782217229979b42a24ea-3cda-49bb-be3f-f98...,2026-06-23 14:24:48,"[{""params"": {}, ""channel"": ""egvp"", ""attachment...",0,sent,0,egvp,2026-06-23 14:24:48


In [11]:
def is_dritt_inside(egvp_intent):
    if "dritt" in egvp_intent['intents']:
        return True
    else:
        return False

In [12]:
egvp_intents['is_dritt_inside'] = egvp_intents.apply(is_dritt_inside, axis=1)

In [14]:
egvp_dritt = egvp_intents[egvp_intents['is_dritt_inside'] == True]

In [15]:
egvp_dritt

,id,ticket_uuid,attachment_id,egvp_id,created_at,intents,ready_for_automation,status,retry,target_type,updated_at,is_dritt_inside
16189,36624,30615c22-3f6e-597f-8410-7a62a4d6603e,68267132,NRW_B21781105441792e9b7ef00-efe0-452e-ad7b-e4b...,2026-06-10 20:52:43,"[{""params"": {""end_page"": null, ""start_page"": n...",1,sent,0,egvp,2026-06-10 20:52:43,True
16191,36626,dd569953-d9a6-51df-9e10-6810223b616f,68267149,NRW_B21781106250166bdf5eed7-537f-4d4f-ac5f-9d8...,2026-06-10 20:52:47,"[{""params"": {""end_page"": 1, ""start_page"": 1, ""...",0,sent,0,egvp,2026-06-10 20:52:47,True
16192,36627,65b16eb0-1e68-5769-a631-2ebc644885fb,68267156,NRW_B217811058667196f3c6e2a-c8b9-478d-a9ae-84e...,2026-06-10 20:52:50,"[{""params"": {""end_page"": null, ""start_page"": n...",1,sent,0,egvp,2026-06-10 20:52:50,True
16211,36646,6f6cf8e8-fafc-5ce7-a5fe-644a455f67ab,68267117,NRW_B217811070158606e4acc92-076f-4635-a48b-244...,2026-06-10 20:56:21,"[{""params"": {""end_page"": null, ""start_page"": n...",1,sent,0,egvp,2026-06-10 20:56:21,True
16212,36647,6f6cf8e8-fafc-5ce7-a5fe-644a455f67ab,68267118,NRW_B217811070158606e4acc92-076f-4635-a48b-244...,2026-06-10 20:56:21,"[{""params"": {""end_page"": null, ""start_page"": n...",1,sent,0,egvp,2026-06-10 20:56:21,True
...,...,...,...,...,...,...,...,...,...,...,...,...
22939,43374,7aba7b38-3db4-5163-af82-7981d007cb55,69376873,NRW_B2178220296234927f0da6e-09d5-4544-b309-56a...,2026-06-23 11:30:50,"[{""params"": {""slug"": ""192210481078"", ""end_page...",0,sent,0,egvp,2026-06-23 11:30:50,True
22943,43378,b0d02ed3-d2fb-5482-b5e7-f51d3f1b8af1,69377076,NRW_B217822018233942593422e-55d4-4289-b898-26f...,2026-06-23 11:30:54,"[{""params"": {""end_page"": null, ""start_page"": n...",1,sent,0,egvp,2026-06-23 11:30:54,True
22953,43388,8735a67b-6a0b-587c-9fe9-9de76d5fff1f,69373508,NRW_B2178220481846206a06859-cfd6-4fc1-a915-16b...,2026-06-23 11:31:16,"[{""params"": {""slug"": ""194208816596"", ""end_page...",1,sent,0,egvp,2026-06-23 11:31:16,True
22957,43392,a25bed45-3bc1-5e0c-aee3-e3c54c9d297b,69374437,NRW_B21782203118574d4e8aa37-e2a9-4b4e-bc62-7c0...,2026-06-23 11:31:22,"[{""params"": {""end_page"": null, ""start_page"": n...",1,sent,0,egvp,2026-06-23 11:31:22,True


In [16]:
import sys
import json
import boto3

sys.path.append("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation")
from utils.prod_utils import get_texts_from_textract_outputs

session = boto3.Session(region_name="eu-central-1", profile_name="739275445236_DataScienceUser")
s3 = session.client("s3")

# 1) Bulk-fetch textract s3_links for all attachment_ids in one query
attachment_ids = egvp_dritt["attachment_id"].dropna().unique().tolist()

textract_df = analytics_db.sql_to_df(
    "SELECT attachment_id, s3_link FROM textract_jobs WHERE attachment_id IN %(ids)s AND status = 'SUCCEEDED'",
    params={"ids": attachment_ids},
)
print(f"Textract jobs found: {len(textract_df)} / {len(attachment_ids)} attachment_ids")

# 2) Download and extract text from each s3_link
def text_from_s3_link(s3_link: str) -> str:
    bucket, key = s3_link.replace("s3://", "").split("/", 1)
    obj = s3.get_object(Bucket=bucket, Key=key)
    blocks = json.loads(obj["Body"].read())
    if isinstance(blocks, dict):
        blocks = blocks.get("Blocks", [])
    return get_texts_from_textract_outputs([blocks])[0]

textract_df["text"] = textract_df["s3_link"].map(text_from_s3_link)

# 3) Merge back onto egvp_dritt
egvp_dritt = egvp_dritt.merge(
    textract_df[["attachment_id", "text"]],
    on="attachment_id",
    how="left",
)
print(f"Rows with text: {egvp_dritt['text'].notna().sum()} / {len(egvp_dritt)}")
egvp_dritt.head()


INFO [2026-06-23 17:02:32] - Found credentials in shared credentials file: ~/.aws/credentials


Textract jobs found: 1160 / 1160 attachment_ids
Rows with text: 1160 / 1160


,id,ticket_uuid,attachment_id,egvp_id,created_at,intents,ready_for_automation,status,retry,target_type,updated_at,is_dritt_inside,text
0,36624,30615c22-3f6e-597f-8410-7a62a4d6603e,68267132,NRW_B21781105441792e9b7ef00-efe0-452e-ad7b-e4b...,2026-06-10 20:52:43,"[{""params"": {""end_page"": null, ""start_page"": n...",1,sent,0,egvp,2026-06-10 20:52:43,True,Obergerichtsvollzieherin K.Deeben\nDienstkonto...
1,36626,dd569953-d9a6-51df-9e10-6810223b616f,68267149,NRW_B21781106250166bdf5eed7-537f-4d4f-ac5f-9d8...,2026-06-10 20:52:47,"[{""params"": {""end_page"": 1, ""start_page"": 1, ""...",0,sent,0,egvp,2026-06-10 20:52:47,True,Peter Büttgen\nMoselstraße 10\nObergerichtsvol...
2,36627,65b16eb0-1e68-5769-a631-2ebc644885fb,68267156,NRW_B217811058667196f3c6e2a-c8b9-478d-a9ae-84e...,2026-06-10 20:52:50,"[{""params"": {""end_page"": null, ""start_page"": n...",1,sent,0,egvp,2026-06-10 20:52:50,True,M. Böckle\nSchragenhofstraße 27\nGerichtsvollz...
3,36646,6f6cf8e8-fafc-5ce7-a5fe-644a455f67ab,68267117,NRW_B217811070158606e4acc92-076f-4635-a48b-244...,2026-06-10 20:56:21,"[{""params"": {""end_page"": null, ""start_page"": n...",1,sent,0,egvp,2026-06-10 20:56:21,True,Sandra Seisenberger\nSchragenhofstr. 27\nOberg...
4,36647,6f6cf8e8-fafc-5ce7-a5fe-644a455f67ab,68267118,NRW_B217811070158606e4acc92-076f-4635-a48b-244...,2026-06-10 20:56:21,"[{""params"": {""end_page"": null, ""start_page"": n...",1,sent,0,egvp,2026-06-10 20:56:21,True,Bundeszentralamt\nfür Steuern\nPOSTANSCHRIFT\n...


In [22]:
import re

# Catch "kein/keinem Ergebnis(se)" phrases, tolerant to extra words in between.
# Examples it matches:
#   - "die Anfrage lieferte kein verwertbares Ergebnis"
#   - "... zu keinem Ergebnis geführt hat"
#   - "Der Abruf hat zu keinem Ergebnis geführt"
#   - "es liegen keine Ergebnisse vor"
KEIN_ERGEBNIS_PATTERN = re.compile(
    r"\bkein(?:em|en|er|es|e)?\b"    # kein / keinem / keinen / keiner / keines / keine
    r"(?:\s+\w+){0,4}?"             # up to 4 optional words in between (e.g. "verwertbares")
    r"\s+ergebniss?(?:e|en|es)?\b", # ergebnis / ergebnisse / ergebnissen / ergebnisses
    flags=re.IGNORECASE,
)

def has_kein_ergebnis(text) -> bool:
    if not isinstance(text, str):
        return False
    return bool(KEIN_ERGEBNIS_PATTERN.search(text))


In [23]:
egvp_dritt["kein_ergebnis"] = egvp_dritt["text"].map(has_kein_ergebnis)

n_match = egvp_dritt["kein_ergebnis"].sum()
n_total = egvp_dritt["text"].notna().sum()
print(f"kein_ergebnis matches: {n_match} / {n_total} ({n_match / n_total:.1%} of dritt=True with text)")

egvp_dritt[egvp_dritt["kein_ergebnis"]].head()


kein_ergebnis matches: 187 / 1160 (16.1% of dritt=True with text)


,id,ticket_uuid,attachment_id,egvp_id,created_at,intents,ready_for_automation,status,retry,target_type,updated_at,is_dritt_inside,text,kein_ergebnis
18,36682,2080d373-3758-53b0-98a2-a0bc70adf87e,68268340,NRW_B21781114877164752c6871-7c0d-4fd3-a56a-f3b...,2026-06-10 20:58:14,"[{""params"": {""end_page"": null, ""start_page"": n...",1,sent,0,egvp,2026-06-10 20:58:14,True,Dieses Dokument ist signiert mit Sign Live! CC...,True
19,36696,e3f94709-e48f-502c-9392-b8ef1e520de7,68269128,NRW_B2178112227842086fd3d9d-8eac-4866-8ac7-7ff...,2026-06-10 22:49:09,"[{""params"": {""end_page"": null, ""start_page"": n...",1,sent,0,egvp,2026-06-10 22:49:09,True,Bundeszentralamt\nfür Steuern\nPOSTANSCHRIFT\n...,True
22,36706,192a95ab-b560-5981-937f-dcdf41712660,68301102,NRW_B2178115074258955906f46-f9c5-4e68-90d8-ba0...,2026-06-11 06:54:26,"[{""params"": {""end_page"": null, ""start_page"": n...",1,sent,0,egvp,2026-06-11 06:54:26,True,"Oliver Thihatmar\nWindstraße 1, 48565 Steinfur...",True
25,36717,8a46c83f-80fc-5a4f-893b-68dc871b879b,68302121,NRW_B217811567586945c6f43c7-06e8-4bc1-b334-0b4...,2026-06-11 07:47:39,"[{""params"": {""end_page"": null, ""start_page"": n...",1,sent,0,egvp,2026-06-11 07:47:39,True,Bundeszentralamt\nfür Steuern\nPOSTANSCHRIFT\n...,True
28,36722,3d057b04-d855-5cd4-a3ea-89114a88042d,68302137,NRW_B21781157184022253eba9b-27d4-40ad-ac2d-f27...,2026-06-11 07:47:45,"[{""params"": {""end_page"": null, ""start_page"": n...",1,sent,0,egvp,2026-06-11 07:47:45,True,Obergerichtsvollzieherin\nAmtsgericht\nBad Seg...,True


In [ ]:
[{"params": {"end_page": null, "start_page": null, "is_invoice_inside": false}, "channel": "egvp", "attachment_id": 68268340, "is_aftercourt": true, "aftercourt_type": "drittauskunft", "full_auto_confidence": true}]

In [26]:
egvp_dritt['is_invoice_inside'] = egvp_dritt['intents'].apply(lambda x: '"is_invoice_inside": false' in x)

In [27]:
egvp_dritt['is_invoice_inside'].value_counts(dropna=False)

is_invoice_inside
True     715
False    445
Name: count, dtype: int64

In [28]:
egvp_dritt_kein_ergebnis_automatable = egvp_dritt[(egvp_dritt['kein_ergebnis'] == True) & (egvp_dritt['is_invoice_inside'] == False)]

In [29]:
egvp_dritt_kein_ergebnis_automatable

,id,ticket_uuid,attachment_id,egvp_id,created_at,intents,ready_for_automation,status,retry,target_type,updated_at,is_dritt_inside,text,kein_ergebnis,is_invoice_inside
55,36883,321ae8a4-b33a-562f-a3dc-27360e0cf0b6,68309627,NRW_B21781179283860e39497b2-7084-4db8-b3bf-b78...,2026-06-11 15:36:26,"[{""params"": {""end_page"": 1, ""start_page"": 1, ""...",0,sent,0,egvp,2026-06-11 15:36:26,True,D. Lindner\nRomain-Rolland-Str. 137\nObergeric...,True,False
61,36908,34e70648-8e5d-5837-84c7-ec6c4cd59744,68310633,NRW_B21781180672833a073f31d-9dfa-485b-a699-504...,2026-06-11 15:37:20,"[{""params"": {""end_page"": 1, ""start_page"": 1, ""...",0,sent,0,egvp,2026-06-11 15:37:20,True,S. Kiser\nMünchner Straße 5\nObergerichtsvollz...,True,False
75,36998,5868c3a1-bdbb-5c32-a5e1-a30843d55549,68312794,NRW_B217811909502026aae6ea8-bf7d-40c0-b550-b3d...,2026-06-11 17:45:56,"[{""params"": {""end_page"": 1, ""start_page"": 1, ""...",0,sent,0,egvp,2026-06-11 17:45:56,True,Obergerichtsvollzieher\nc/o AG Bochum Josef-Ne...,True,False
78,37010,70755d4b-8498-5b35-9671-0e49ce64a0c0,68312737,NRW_B2178119035984188e0022e-401e-44d4-95c5-e7b...,2026-06-11 17:46:19,"[{""params"": {""end_page"": 1, ""start_page"": 1, ""...",0,sent,0,egvp,2026-06-11 17:46:19,True,Björn Ellendt\nStraße am Hochwald 21\nObergeri...,True,False
79,37011,985558dd-26f3-5952-8f9f-0e93b0dac0bc,68312754,NRW_B21781190257355284bd207-d94c-4589-8f70-9f6...,2026-06-11 17:46:20,"[{""params"": {""end_page"": 1, ""start_page"": 1, ""...",0,sent,0,egvp,2026-06-11 17:46:20,True,Obergerichtsvollzieher\nBüroanschrift\nSevdi\n...,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1113,42952,03f4b66e-75a5-5f54-b6b4-8c10bc6c09ee,69346380,NRW_B217821247590649a57a059-ea69-4fc5-b79c-59f...,2026-06-22 20:31:34,"[{""params"": {""slug"": ""128034968965"", ""end_page...",0,sent,0,egvp,2026-06-22 20:31:34,True,Gerichtsvollzieherin\nFrankfurter Str. 130\nKe...,True,False
1126,43040,10647790-c422-572b-90cd-9ea957673848,69348288,NRW_B2178212685311533d7dc53-7b6e-4f13-a8dc-b60...,2026-06-22 20:44:14,"[{""params"": {""slug"": ""124817455551"", ""end_page...",0,sent,0,egvp,2026-06-22 20:44:14,True,Gerichtsvollzieherin\nFrankfurter Str. 130\nKe...,True,False
1135,43142,0d5de14e-d641-50e7-b111-09f687b8e9b3,69343514,NRW_B21782121984029f8d19f33-ee88-4810-9ec2-c58...,2026-06-22 21:43:13,"[{""params"": {""slug"": ""197467796300"", ""end_page...",0,sent,0,egvp,2026-06-22 21:43:13,True,Anja Fischer\nSchwieberdinger Strasse 79\nGeri...,True,False
1140,43183,4ea74d89-d006-520e-aa97-acc50678b302,69354607,NRW_B21782161905751635d7ee9-7bc8-4488-b8d0-414...,2026-06-22 22:49:27,"[{""params"": {""slug"": ""185930801922"", ""end_page...",0,sent,0,egvp,2026-06-22 22:49:27,True,Obergerichtsvollzieher\nPostanschrift:\nMichae...,True,False


In [30]:
dritt_automatable = egvp_dritt_kein_ergebnis_automatable.shape[0]
dritt_kein_ergebnis = egvp_dritt[egvp_dritt['kein_ergebnis'] == True].shape[0]
dritt_total = egvp_dritt.shape[0]


In [35]:
print(f"Total dritt: {dritt_total}")
print(f"Total dritt with kein_ergebnis: {dritt_kein_ergebnis}, prct: {dritt_kein_ergebnis / dritt_total:.1%}")
print(f"Total dritt with kein_ergebnis and is_invoice_inside=False: {dritt_automatable}, prct: {dritt_automatable / dritt_total:.1%}")

Total dritt: 1160
Total dritt with kein_ergebnis: 187, prct: 16.1%
Total dritt with kein_ergebnis and is_invoice_inside=False: 84, prct: 7.2%
